In [29]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [30]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-03 17:43:11|Manhattan      |10.56        |4.5         |53                 |
|2026-05-03 17:43:11|Manhattan      |2.48         |4.5         |54                 |
|2026-05-03 17:43:11|Manhattan      |10.56        |4.5         |54                 |
|2026-05-03 17:43:11|Manhattan      |10.56        |0.6         |54                 |
|2026-05-03 17:43:11|Manhattan      |10.56        |1.3         |53                 |
|2026-05-03 17:43:11|Queens         |36.66        |0.7         |54                 |
|2026-05-03 17:43:11|Manhattan      |10.56        |1.3         |54                 |
|2026-05-03 17:43:11|Queens         |0.0          |2.9         |53                 |
|2026-05-03 17:43:11|Manhattan      |10.56        |3.5         |5

In [31]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Brooklyn       |3088         |12.6         |2.67    |54          |
|Manhattan      |11094        |12.65        |1.81    |54          |
|Bronx          |5094         |21.34        |1.6     |54          |
|Queens         |4100         |22.55        |26.74   |54          |
|Staten Island  |1444         |36.86        |3.32    |54          |
+---------------+-------------+-------------+--------+------------+



In [32]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|        18833|
|     Brooklyn|         6108|
|Staten Island|        13234|
|    Manhattan|        13237|
|        Bronx|        12216|
+-------------+-------------+



In [33]:
# Borough-level diagnostics: traffic-only vs traffic+AQ vs traffic+AQ+weather
spark.sql("""
WITH traffic_only AS (
  SELECT borough AS traffic_borough, COUNT(*) AS traffic_rows
  FROM parquet.`s3a://raw-data/traffic/`
  WHERE borough IS NOT NULL
  GROUP BY borough
),
aq_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_rows
  FROM local.db.enriched_traffic
  GROUP BY traffic_borough
),
aq_weather_joined AS (
  SELECT traffic_borough, COUNT(*) AS traffic_aq_weather_rows
  FROM local.db.enriched_traffic
  WHERE weather_temperature IS NOT NULL
  GROUP BY traffic_borough
)
SELECT
  t.traffic_borough,
  t.traffic_rows,
  COALESCE(a.traffic_aq_rows, 0) AS traffic_aq_rows,
  COALESCE(w.traffic_aq_weather_rows, 0) AS traffic_aq_weather_rows,
  ROUND(100.0 * COALESCE(a.traffic_aq_rows, 0) / t.traffic_rows, 2) AS aq_match_pct,
  ROUND(100.0 * COALESCE(w.traffic_aq_weather_rows, 0) / t.traffic_rows, 2) AS aq_weather_match_pct
FROM traffic_only t
LEFT JOIN aq_joined a ON t.traffic_borough = a.traffic_borough
LEFT JOIN aq_weather_joined w ON t.traffic_borough = w.traffic_borough
ORDER BY t.traffic_rows DESC
""").show(truncate=False)

+---------------+------------+---------------+-----------------------+------------+--------------------+
|traffic_borough|traffic_rows|traffic_aq_rows|traffic_aq_weather_rows|aq_match_pct|aq_weather_match_pct|
+---------------+------------+---------------+-----------------------+------------+--------------------+
|Queens         |18833       |4100           |3220                   |21.77       |17.10               |
|Manhattan      |13237       |11094          |10360                  |83.81       |78.27               |
|Staten Island  |13234       |1444           |840                    |10.91       |6.35                |
|Bronx          |12216       |5094           |4480                   |41.70       |36.67               |
|Brooklyn       |6108        |3088           |2800                   |50.56       |45.84               |
+---------------+------------+---------------+-----------------------+------------+--------------------+



In [34]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-03 17:07:54.674|2026-05-05 21:00:34.664|
+-----------------------+-----------------------+

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-01 16:18:19.567|2026-05-05 21:01:12.309|
+-----------------------+-----------------------+



In [35]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+-----+
|min(traffic_event_ts)|max(traffic_event_ts)|total|
+---------------------+---------------------+-----+
|2026-05-01 14:38:00  |2026-05-03 17:43:11  |24820|
+---------------------+---------------------+-----+



In [36]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-03 17:43:11|2026-05-01 16:18:19.567|
+-------------------+-----------------------+



In [37]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |445       |1442                      |
|Brooklyn       |154       |874                       |
|Manhattan      |149       |874                       |
|Brooklyn       |148       |874                       |
|Brooklyn       |157       |874                       |
|Manhattan      |448       |874                       |
|Manhattan      |106       |874                       |
|Manhattan      |124       |874                       |
|Manhattan      |221       |732                       |
|Manhattan      |364       |732                       |
|Manhattan      |215       |732                       |
|Manhattan      |223       |590                       |
|Manhattan      |217       |590                       |
|Manhattan      |145       |590                       |
|Manhattan      |150       |590                 

In [38]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+----+
|h3_index       |rows|
+---------------+----+
|872a10729ffffff|4370|
|872a100aeffffff|4050|
|872a1072dffffff|2360|
|872a100d2ffffff|2196|
|872a1008bffffff|1748|
|872a100f2ffffff|1530|
|872a1072cffffff|1442|
|872a10623ffffff|984 |
|872a1000bffffff|656 |
|872a1001effffff|656 |
+---------------+----+



In [39]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic,
  MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq,
  MAX(aq_event_ts) AS max_aq,
  MIN(weather_event_ts) AS min_weather,
  MAX(weather_event_ts) AS max_weather
FROM local.db.enriched_traffic
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_rows,
  SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) AS weather_rows
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-----------------------+-----------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq                 |max_aq                 |min_weather        |max_weather        |
+-------------------+-------------------+-----------------------+-----------------------+-------------------+-------------------+
|2026-05-01 14:38:00|2026-05-03 17:43:11|2026-05-01 16:18:19.194|2026-05-03 17:29:28.787|2026-05-03 17:14:18|2026-05-03 17:30:29|
+-------------------+-------------------+-----------------------+-----------------------+-------------------+-------------------+

+----------+-------+------------+
|total_rows|aq_rows|weather_rows|
+----------+-------+------------+
|24820     |22028  |21700       |
+----------+-------+------------+



In [40]:
# AQ Debugging

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_matched_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_match_pct
FROM local.db.enriched_traffic
""").show(truncate=False)

+----------+---------------+------------+
|total_rows|aq_matched_rows|aq_match_pct|
+----------+---------------+------------+
|24820     |22028          |88.75       |
+----------+---------------+------------+



In [41]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic, MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq, MAX(aq_event_ts) AS max_aq
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-----------------------+-----------------------+
|min_traffic        |max_traffic        |min_aq                 |max_aq                 |
+-------------------+-------------------+-----------------------+-----------------------+
|2026-05-01 14:38:00|2026-05-03 17:43:11|2026-05-01 16:18:19.194|2026-05-03 17:29:28.787|
+-------------------+-------------------+-----------------------+-----------------------+



In [42]:
spark.sql("""
SELECT
  traffic_borough,
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_non_null_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_non_null_pct
FROM local.db.enriched_traffic
GROUP BY traffic_borough
ORDER BY traffic_borough;
""").show(truncate=False)

+---------------+----------+----------------+---------------+
|traffic_borough|total_rows|aq_non_null_rows|aq_non_null_pct|
+---------------+----------+----------------+---------------+
|Bronx          |5094      |4558            |89.48          |
|Brooklyn       |3088      |2840            |91.97          |
|Manhattan      |11094     |10512           |94.75          |
|Queens         |4100      |3266            |79.66          |
|Staten Island  |1444      |852             |59.00          |
+---------------+----------+----------------+---------------+

